# Export Hotel Pairs to CSV

This notebook reads the complete hotel pairs data from the Delta Lake path and exports it to a single CSV file for external analysis.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round as spark_round, min as spark_min, max as spark_max
import os

# Override any problematic environment variables
os.environ.pop('HADOOP_CONF_DIR', None)
os.environ.pop('YARN_CONF_DIR', None)

# Create Spark session with MinIO/S3A configuration
# Simplified without Delta Lake dependencies - read as parquet directly
spark = SparkSession.builder \
    .appName("Analyze Hotel Pairs") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.send.buffer", "8192") \
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "8192") \
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3") \
    .config("spark.hadoop.fs.s3a.retry.limit", "3") \
    .config("spark.hadoop.fs.s3a.retry.interval", "500") \
    .config("spark.hadoop.fs.s3a.retry.throttle.limit", "3") \
    .config("spark.hadoop.fs.s3a.retry.throttle.interval", "1000") \
    .config("spark.hadoop.fs.s3a.connection.maximum", "50") \
    .config("spark.hadoop.fs.s3a.threads.max", "10") \
    .config("spark.hadoop.fs.s3a.threads.core", "5") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.max.total.tasks", "5") \
    .config("spark.hadoop.fs.s3a.readahead.range", "65536") \
    .config("spark.hadoop.fs.s3a.paging.maximum", "5") \
    .config("spark.hadoop.fs.s3a.list.version", "2") \
    .config("spark.hadoop.fs.s3a.committer.threads", "4") \
    .config("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/staging") \
    .config("spark.hadoop.fs.s3a.buffer.dir", "/tmp") \
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
    .config("spark.hadoop.fs.s3a.multipart.threshold", "2147483647") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400") \
    .config("spark.hadoop.fs.s3a.fast.upload", "true") \
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "disk") \
    .config("spark.hadoop.fs.s3a.fast.upload.active.blocks", "4") \
    .config("spark.hadoop.fs.s3a.block.size", "33554432") \
    .config("spark.hadoop.fs.s3a.metadatastore.authoritative", "false") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.port", "4060") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✓ Spark session created successfully")
print("✓ Spark UI available at: http://localhost:4060")

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/nakul.patil/.ivy2.5.2/cache
The jars for the packages stored in: /Users/nakul.patil/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-52cac05e-25bd-48a3-84e8-e72dc3495ac0;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 112ms :: artifacts dl 3ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final

✓ Spark session created successfully
✓ Spark UI available at: http://localhost:4060


In [8]:
# Read hotel pairs data from the Parquet source
delta_path = "s3a://delta-lake-mumbai/bronze/hotel_pairs/"
print(f"Reading data from: {delta_path}")

df = spark.read.format("parquet").load(delta_path)

print(f"✓ Data loaded successfully with {df.count():,} rows and {len(df.columns)} columns.")

Reading data from: s3a://delta-lake-mumbai/bronze/hotel_pairs/
✓ Data loaded successfully with 27,264 rows and 39 columns.


In [9]:

# Load hotel master data and join contact_address_line1 for both _i and _j entities
hotels_path = "s3a://delta-lake-mumbai/bronze/hotels/"
print(f"Reading hotel master data from: {hotels_path}")

hotels = spark.read.format("parquet").load(hotels_path)
print(f"✓ Hotels loaded: {hotels.count():,} rows, columns: {hotels.columns}")

# Select only the fields we need from the hotels table
hotels_slim = hotels.select("providerHotelId", "contact_address_line1")

# Join for _i side
df = df.join(
    hotels_slim.withColumnRenamed("providerHotelId", "_join_i")
               .withColumnRenamed("contact_address_line1", "address_line1_i"),
    df["providerHotelId_i"] == col("_join_i"),
    how="left"
).drop("_join_i")

# Join for _j side
df = df.join(
    hotels_slim.withColumnRenamed("providerHotelId", "_join_j")
               .withColumnRenamed("contact_address_line1", "address_line1_j"),
    df["providerHotelId_j"] == col("_join_j"),
    how="left"
).drop("_join_j")

print(f"✓ Joined contact_address_line1 for both _i and _j. Total columns: {len(df.columns)}")
df.select("providerHotelId_i", "address_line1_i", "providerHotelId_j", "address_line1_j").show(5, truncate=False)


Reading hotel master data from: s3a://delta-lake-mumbai/bronze/hotels/
✓ Hotels loaded: 2,584 rows, columns: ['id', 'name', 'relevanceScore', 'providerId', 'providerHotelId', 'providerName', 'language', 'geoCode', 'contact', 'type', 'category', 'starRating', 'distance', 'attributes', 'imageCount', 'availableSuppliers', 'geoCode_lat', 'geoCode_long', 'contact_address_line1', 'contact_address_city_name', 'contact_address_state_name', 'contact_address_country_name', 'contact_address_country_code', 'contact_address_postalCode', 'contact_phones', 'contact_fax', 'contact_emails', 'processing_time_utc', 'combined_address', 'uid', 'normalized_name', 'geohash', 'name_embedding', 'normalized_name_embedding', 'address_embedding']
✓ Joined contact_address_line1 for both _i and _j. Total columns: 41
+-----------------+----------------------------------------------------------+-----------------+----------------------------------------------------------------------------------------------+
|providerH

In [10]:
# Define the output path for the CSV file
output_path = "/Users/nakul.patil/Documents/hotel-mapping/output/hotel_pairs.csv"
print(f"Writing data to CSV file at: {output_path}")

# Use coalesce(1) to force writing to a single file
# Use mode("overwrite") to replace the file if it already exists
df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)

print(f"✓ Successfully exported data to CSV.")
print("Note: The CSV file is located inside a directory with a name like 'part-00000-....csv'")

Writing data to CSV file at: /Users/nakul.patil/Documents/hotel-mapping/output/hotel_pairs.csv
✓ Successfully exported data to CSV.
Note: The CSV file is located inside a directory with a name like 'part-00000-....csv'


26/03/05 02:19:22 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 907246 ms exceeds timeout 120000 ms
26/03/05 02:19:22 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/05 02:19:25 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$